See the readme for information about adding a dataset.

Make the NOBS database directly from the data sources.

The date ranges of the two variables are **wildly** diffeent, by decades in some cases.

Therefore I decided to create a nobs table for each variable.

Which also means new variables can be added without recreating the existing variable NOBS tables.

In [3]:
import pandas as pd
import json
import constants

In [4]:
with open('sites.json') as stream:
    sites = json.load(stream)

In [5]:
counts = {}
for variable in sites:
    print(variable)
    platform_counts = None
    name = sites[variable]['short_names'][0]
    for dataset in sites[variable]['datasets']:
        site_codes = pd.read_csv(dataset+'.csv?site_code&distinct()')
        for i,row in site_codes.iterrows():
            count_url = dataset + '.csv?' + name +',time,site_code' + '&site_code="'+row["site_code"]+'"&orderByCount(\"site_code,time/1month\")'
            print(count_url)
            count_df = pd.read_csv(count_url, skiprows=[1])
            if platform_counts is not None:
                platform_counts = pd.concat([platform_counts, count_df])
            else:
                platform_counts = count_df
                
    counts[variable] = platform_counts

temperature
https://data.pmel.noaa.gov/pmel/erddap/tabledap/keo_hourly.csv?TEMP,time,site_code&site_code="KEO"&orderByCount("site_code,time/1month")
https://data.pmel.noaa.gov/pmel/erddap/tabledap/papa_hourly_temp.csv?TEMP,time,site_code&site_code="Papa"&orderByCount("site_code,time/1month")
https://data.pmel.noaa.gov/pmel/erddap/tabledap/pirata_hourly_temp.csv?TEMP,time,site_code&site_code="0n23w"&orderByCount("site_code,time/1month")
https://data.pmel.noaa.gov/pmel/erddap/tabledap/pirata_hourly_temp.csv?TEMP,time,site_code&site_code="0n35w"&orderByCount("site_code,time/1month")
https://data.pmel.noaa.gov/pmel/erddap/tabledap/pirata_hourly_temp.csv?TEMP,time,site_code&site_code="10s10w"&orderByCount("site_code,time/1month")
https://data.pmel.noaa.gov/pmel/erddap/tabledap/pirata_hourly_temp.csv?TEMP,time,site_code&site_code="12n23w"&orderByCount("site_code,time/1month")
https://data.pmel.noaa.gov/pmel/erddap/tabledap/pirata_hourly_temp.csv?TEMP,time,site_code&site_code="14s32w"&orderBy

In [7]:
for variable in counts:
    with constants.postgres_engine.connect() as conn:
        counts[variable].to_sql(variable+'_nobs', con=conn, index=False, if_exists='replace')